This file is part of PriorTiltDecoder.

Copyright (c) 2026 IonQ, Inc., all rights reserved

Licensed under the Creative Commons Attribution-NonCommercial-ShareAlike
4.0 International License (CC BY-NC-SA 4.0).

You may obtain a copy of the License at:
https://creativecommons.org/licenses/by-nc-sa/4.0/

# Reproducing the results of the PriorTilt paper

*PriorTilt: Decoding quantum LDPC codes with sparse perturbations of belief propagation priors*, Min Ye and Nicolas Delfosse.

Before running this notebook, install the dependencies and build the decoder (see `README.md`):

```bash
pip install -r requirements.txt
pip install ./decoder
```

Three codes are benchmarked (the `code` argument of every function below accepts these names):

| `code` | memory experiment | noise rates `p_CNOT` |
|---|---|---|
| `"BB144"` | BB [[144,12,12]], 12 syndrome extraction rounds | 0.001, 0.002, 0.003, 0.004, 0.005 |
| `"BB70"` | BB [[70,6,9]], 9 rounds | 0.0005, 0.0008, 0.001, 0.002 |
| `"GB102"` | GB [[102,22,9]], 9 rounds | 0.0005, 0.0008, 0.001, 0.002 |

The figure of merit is the logical error rate (LER) per syndrome extraction cycle, `(failures / shots) / syndrome_rounds`.

## 1. The paper's figure from the stored data

`simulation_results/paper_data/` contains the raw results behind every point of the figure, for all three decoders.
`plot_paper_figure()` draws the three panels (one curve per decoder: the sum of the X- and Z-memory LERs per cycle);
`sum_xz=False` shows the X and Z curves separately. `savefig="paper_figure.pdf"` writes the figure to a file.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from simulation_functions import plot_paper_figure, plot_logical_error_rate, print_table, load_paper_data

fig = plot_paper_figure(sum_xz=True)
plt.show()

`print_table(code)` prints the numbers behind one panel (per basis and x+z: the LER of each decoder, the PriorTilt / Relay-BP
and PriorTilt / beam-search ratios and the failure counts), and `load_paper_data()` returns the tidy table of all points.

In [ ]:
print_table("BB144")
load_paper_data("BB144").head()

## 2. Function `one_point_simulation`: re-simulate one point with PriorTilt

```python
def one_point_simulation(code="BB144", p_CNOT=0.005, basis="z", maximum_shots=1_000_000, maximum_errors=100, max_rounds=3000, num_workers=None)
```

Simulates the memory experiment of `code` at `p_CNOT` with PriorTilt in the three-phase mode with the schedule fitted
for that point (exactly the paper's configuration: one solution, 60 plain BP iterations, tilt rounds of 6 iterations, at most
`max_rounds` = 3000 rounds), until `maximum_errors` logical failures or `maximum_shots` shots per basis, and prints the LER
next to the paper's value for the same point. `basis` is `"z"` (Z memory), `"x"` (X memory) or `"xz"` (both; the printed x + z
sum is what the paper's figure shows). `(code, p_CNOT)` must be one of the pairs in the table above.
sinter prints its progress (shots and failures collected so far) below the cell while the simulation runs.

⚠️ The defaults, BB144 Z memory at `p_CNOT = 0.005`, take about a minute on a laptop with 11 worker processes (about 4,000 shots
for 100 failures); `basis="xz"` doubles that. `p_CNOT = 0.004` takes about 3x longer (34,000 shots per basis) and `p_CNOT = 0.003`
about 6x longer (320,000 shots per basis). The paper's points at `p_CNOT = 0.001` needed 7–9·10^8 shots per basis: use
`full_simulation` (section 3) for those, and expect a very long run.

In [ ]:
from simulation_functions import one_point_simulation

result = one_point_simulation(code="BB144", p_CNOT=0.005, basis="z", maximum_shots=1_000_000, maximum_errors=100)
result

## 3. Function `full_simulation`: re-simulate a whole PriorTilt curve

```python
def full_simulation(code, maximum_shots=1_000_000_000, maximum_errors=100, rates=None, num_workers=None, max_rounds=3000)
```

Runs PriorTilt on every noise rate of `code` (or only on `rates`), both bases, until `maximum_errors` failures or
`maximum_shots` shots per point, and writes the results in sinter's CSV format (with the decoder parameters of each row) to
`simulation_results/<code>_PriorTilt_3phase.csv`. The run is resumable: existing rows of that file are reused and only the
missing shots are collected.

⚠️ **This is a very long computation.** The paper's points at the lowest noise rate of each code used up to 10^9 shots per
basis, i.e. hundreds of CPU-days per code on a cluster. In the cell below `rates` is restricted to the highest noise rate and
`maximum_shots` is small so that the cell finishes in minutes; a small number of shots gives a large statistical variance.
To reproduce the paper's curves, remove the `rates` restriction and set `maximum_shots` to at least 10^9.

In [ ]:
from simulation_functions import full_simulation

full_simulation(code="BB70", rates=[0.002], maximum_shots=200_000, maximum_errors=100)

`plot_logical_error_rate(code)` draws one panel of the paper's figure and, with `own_results=True` (the default), adds the
points of your own `full_simulation(code)` run as hollow black markers for comparison.

In [ ]:
plot_logical_error_rate("BB70", sum_xz=True, own_results=True)
plt.show()

## 4. Function `decoding_time`: single-thread decoding time of PriorTilt

```python
def decoding_time(code="BB144", p_CNOT=0.001, num_shots=10_000, max_rounds=3000)
```

Decodes `num_shots` sampled shots of the Z memory experiment one at a time on a single thread and prints the mean and the 99th
percentile of the per-shot wall time, the two statistics reported in the paper. The 99th percentile rests on `num_shots / 100`
shots; 10^4 shots give a stable mean and a 99th percentile good to a few percent.

In [ ]:
from simulation_functions import decoding_time

decoding_time(code="BB144", p_CNOT=0.001, num_shots=10_000)

## 5. Functions `offset_fraction` and `offset_fraction_table`: how many bits are kicked per tilt round

```python
def offset_fraction(code, p_CNOT, basis="xz", num_shots=None, num_workers=None, max_rounds=3000, seed=7)
def offset_fraction_table(codes=("BB70",), rates=(0.001, 0.002), num_shots=None, num_workers=None)
```

The paper reports, for each code at `p_CNOT` = 0.001 and 0.002, the fraction of bits (error mechanisms) that receive a nonzero
prior offset per tilt round, with the three-phase schedule and the X and Z memory experiments pooled. The decoder keeps
diagnostic counters for this (`PriorTilt.tilt_statistics()`): only shots that are still unconverged after the plain BP phase
enter the tilt rounds, and at the end of every tilt round the bits whose LLR sum is below `threshold_factor` × prior get a
nonzero offset. The fraction is (number of (bit, tilt round) pairs with a nonzero offset) / (number of tilt rounds × number
of bits), summed over all contributing shots. `offset_fraction` measures one (code, `p_CNOT`) cell and prints it next to the
paper's value; `offset_fraction_table` runs several cells and prints them as in the paper's table. The paper's rows (per basis,
with the per-shot 90th percentile and maximum) are in `simulation_results/paper_data/offset_fraction.csv`
(`load_paper_offset_fraction()`).

The sample sizes are the paper's, 100,000 shots per basis at `p_CNOT` = 0.001 and 30,000 at 0.002. The default below, BB70 at
both rates, takes about half a minute on a laptop with 11 worker processes; the paper's full table is
`offset_fraction_table(codes=("BB144", "BB70", "GB102"))` and takes about 4x longer. New shots and decoder seeds are drawn, so
the agreement with the paper is statistical: within about 10% for BB144 and BB70, about 20% for GB102 at `p_CNOT` = 0.001,
where fewer than 100 shots per basis reach the tilt phase.

In [ ]:
from simulation_functions import offset_fraction_table, load_paper_offset_fraction

# the paper's full table: offset_fraction_table(codes=("BB144", "BB70", "GB102"))
table = offset_fraction_table()
table

## 6. XYZ decoding of BB144 at `p_CNOT` = 0.003

The paper also decodes BB144 at `p_CNOT` = 0.003 on the *XYZ* circuits: the same memory experiments, but after the first
round every detector compares **all** stabiliser measurements (X and Z), so the decoder works on the full detector error model
(about 68,000 error mechanisms instead of 8,784) and can use the correlations between X and Z errors. The two circuits are
`StimCircuit/BB[[144,12,12]],xyz_memory_{X,Z},error_rate=0.003,syndrome_rounds=12.stim`. PriorTilt uses its own three-phase
schedule for this model (`priortilt.params.xyz_schedule()`: ends [50, 420], phases (0.07, 0.5, 0.8, 2.0), (0.1372, 0.5, 0.784,
0.686), (0.049, 0.5, 1.568, 1.372)), one solution and a budget of 6000 tilt rounds. The paper's results, with the Relay-BP
reference (601 legs, 5 solutions), are stored in `simulation_results/paper_data/BB144_XYZ_p0.003.csv`:

```python
def load_paper_xyz(code="BB144", p_CNOT=0.003)   # the stored rows as a table
def print_xyz_table(code="BB144", p_CNOT=0.003)  # the paper's table
```

In [ ]:
from simulation_functions import print_xyz_table, load_paper_xyz

xyz = print_xyz_table()

⚠️ **Re-simulating the XYZ point is not a laptop-scale computation.** A shot takes about 0.6 s on a laptop core, and the paper's 100
failures per basis needed 1.6 million shots, hundreds of CPU-hours per basis (9.4 hours on 238 cores for both bases). A few
thousand shots on a laptop will most likely see zero failures at this LER (5·10^-6 per cycle, 6·10^-5 per shot). The cell below
is therefore commented out so that "Run All" skips it; uncomment it to try, and bound the run with `maximum_shots`.

```python
def xyz_one_point_simulation(basis="z", maximum_shots=2_000_000, maximum_errors=100, code="BB144", p_CNOT=0.003, max_rounds=6000, num_workers=None)
```

In [ ]:
# from simulation_functions import xyz_one_point_simulation
#
# # 2,000 shots take a few minutes with 11 worker processes and will most likely show 0 failures;
# # the paper's point needed 1.6 million shots per basis.
# xyz_result = xyz_one_point_simulation(basis="z", maximum_shots=2_000, maximum_errors=100)
# xyz_result

## 7. Using the decoder directly

`PriorTilt` works on any `stim.DetectorErrorModel`. With `code=` and `p=` it uses the paper's parameters; for another code or
circuit the parameters must be tuned first and passed explicitly (`schedule=` or the four scalar parameters), because the
logical error rate depends strongly on them.

In [ ]:
import stim
from priortilt import PriorTilt
from simulation_functions import load_circuit

circuit = load_circuit("GB102", 0.002, "z")
decoder = PriorTilt(circuit.detector_error_model(), code="GB102", p=0.002)
print(decoder)
dets, obs = circuit.compile_detector_sampler(seed=1).sample(2000, separate_observables=True)
predictions = decoder.decode_batch(dets)
print((predictions != obs).any(axis=1).sum(), "logical failures in 2000 shots")